In [1]:
# ============================================================================
# CE_Avg vs IPA_Avg — compare the two max-2nd-derivative IPA methods
#
#   CE_Avg_method : average CE over runs -> one elbow -> one IPA
#   IPA_Avg_method: one elbow per run   -> one IPA per run -> <IPA>, STD
#
# Reads the two summary CSVs (run both method notebooks first), then:
#   1. Comparison CSV per (P%, BS): both IPAs, both BN_learned, the difference
#      Diff = IPA_Avg - CE_Avg, relative difference (% of CE_Avg) and
#      Diff / STD (difference in units of the run-to-run spread)
#   2. Plot: IPA vs P%, one panel per BS, both methods overlaid
#   3. Plot: relative difference vs P%, and BN_learned vs P% for both methods
#
# Requires: numpy, pandas, matplotlib
# ============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ── Config ────────────────────────────────────────────────────────────────────
BATCH_SIZES = [64, 50000]
BS_COLOR    = {64: "#1f77b4", 50000: "#2ca02c"}
CE_AVG_COLOR = "#444444"

# Paths are relative to this notebook: SLP-CIFAR10/IPA_2nd_derivative_method/Comparison/
CE_AVG_CSV  = os.path.abspath(os.path.join("..", "CE_Avg_method", "ipa_summary_2nd_deriv_standalone.csv"))
IPA_AVG_CSV = os.path.abspath(os.path.join("..", "IPA_Avg_method", "ipa_summary_2nd_deriv_ipa_avg.csv"))
OUT_DIR     = os.path.abspath(".")
# ──────────────────────────────────────────────────────────────────────────────

for path in (CE_AVG_CSV, IPA_AVG_CSV):
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} — run that method's notebook first")
ce_df  = pd.read_csv(CE_AVG_CSV)
ipa_df = pd.read_csv(IPA_AVG_CSV)
ce_df["P%"]  = ce_df["P%"].round(6)
ipa_df["P%"] = ipa_df["P%"].round(6)
m = ce_df.merge(ipa_df, on="P%", how="outer", suffixes=("_ce", "_ipa")).sort_values("P%")

# ── Comparison table ──────────────────────────────────────────────────────────
cmp_df = pd.DataFrame({"P%": m["P%"].values})
for bs in BATCH_SIZES:
    ce_ipa  = m[f"IPA_Avg_{bs}_ce"].values
    ipa_ipa = m[f"IPA_Avg_{bs}_ipa"].values
    std     = m[f"STD_{bs}"].values
    diff    = ipa_ipa - ce_ipa
    cmp_df[f"BN_CE_Avg_{bs}"]       = m[f"BN_learned_{bs}"].values
    cmp_df[f"BN_IPA_Avg_{bs}"]      = m[f"BN_learned_mean_{bs}"].values
    cmp_df[f"IPA_CE_Avg_{bs}"]      = ce_ipa
    cmp_df[f"IPA_IPA_Avg_{bs}"]     = ipa_ipa
    cmp_df[f"STD_{bs}"]             = std
    cmp_df[f"Diff_{bs}"]            = diff
    cmp_df[f"RelDiff_pct_{bs}"]     = 100 * diff / ce_ipa
    cmp_df[f"Diff_over_STD_{bs}"]   = diff / std

csv_path = os.path.join(OUT_DIR, "ipa_compare_ce_avg_vs_ipa_avg.csv")
cmp_df.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")
print(cmp_df[["P%"] + [c for b in BATCH_SIZES
                       for c in (f"IPA_CE_Avg_{b}", f"IPA_IPA_Avg_{b}", f"RelDiff_pct_{b}", f"Diff_over_STD_{b}")]]
      .to_string(index=False, float_format=lambda x: f"{x:.4g}"))

# ── Plot 1: IPA vs P%, one panel per BS ───────────────────────────────────────
plt.rcParams.update({"font.size": 12})
fig, axes = plt.subplots(1, len(BATCH_SIZES), figsize=(16, 5), sharey=True)
for ax, bs in zip(axes, BATCH_SIZES):
    sub = cmp_df.dropna(subset=[f"IPA_CE_Avg_{bs}", f"IPA_IPA_Avg_{bs}"])
    ax.plot(sub["P%"], sub[f"IPA_CE_Avg_{bs}"], "o-", color=CE_AVG_COLOR,
            ms=5, lw=1.5, label="CE_Avg  (elbow of <CE>)")
    ax.errorbar(sub["P%"], sub[f"IPA_IPA_Avg_{bs}"], yerr=sub[f"STD_{bs}"], fmt="^-.",
                color=BS_COLOR[bs], ms=6, lw=2, capsize=3, label="IPA_Avg  (<IPA> ± STD)")
    ax.set_title(f"BS={bs}")
    ax.set_xlabel("Pruning Percentage (%)")
    ax.grid(True, alpha=0.3)
    ax.legend(frameon=False, fontsize=10)
axes[0].set_ylabel("IPA")
fig.suptitle("IPA vs Pruning — CE_Avg vs IPA_Avg (max 2nd derivative elbow)")
png_path = os.path.join(OUT_DIR, "ipa_compare_methods.png")
plt.tight_layout()
plt.savefig(png_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {png_path}")

# ── Plot 2: relative difference and BN_learned vs P% ──────────────────────────
fig, (ax_d, ax_bn) = plt.subplots(1, 2, figsize=(15, 5))
for bs in BATCH_SIZES:
    sub = cmp_df.dropna(subset=[f"RelDiff_pct_{bs}"])
    ax_d.plot(sub["P%"], sub[f"RelDiff_pct_{bs}"], "^-.", color=BS_COLOR[bs],
              ms=6, lw=2, label=f"BS={bs}")
    ax_bn.plot(sub["P%"], sub[f"BN_CE_Avg_{bs}"], "o-", color=BS_COLOR[bs],
               ms=5, lw=1.5, label=f"BS={bs} CE_Avg")
    ax_bn.plot(sub["P%"], sub[f"BN_IPA_Avg_{bs}"], "^-.", color=BS_COLOR[bs],
               ms=6, lw=2, alpha=0.7, label=f"BS={bs} IPA_Avg (mean)")
ax_d.axhline(0, color="k", lw=0.8)
ax_d.set_xlabel("Pruning Percentage (%)")
ax_d.set_ylabel("(IPA_Avg − CE_Avg) / CE_Avg  (%)")
ax_d.set_title("Relative difference between methods")
ax_d.grid(True, alpha=0.3)
ax_d.legend(frameon=False)
ax_bn.set_xlabel("Pruning Percentage (%)")
ax_bn.set_ylabel("BN_learned")
ax_bn.set_title("Elbow location")
ax_bn.grid(True, alpha=0.3)
ax_bn.legend(frameon=False, fontsize=9, ncol=2)
png_path = os.path.join(OUT_DIR, "ipa_compare_differences.png")
plt.tight_layout()
plt.savefig(png_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {png_path}")

Saved: C:\Users\lpand003\Desktop\Research\ipa-pruning\physlab\SLP\SLP-CIFAR10\IPA_2nd_derivative_method\Comparison\ipa_compare_ce_avg_vs_ipa_avg.csv
 P%  IPA_CE_Avg_64  IPA_IPA_Avg_64  RelDiff_pct_64  Diff_over_STD_64  IPA_CE_Avg_50000  IPA_IPA_Avg_50000  RelDiff_pct_50000  Diff_over_STD_50000
  0        0.03782         0.01783          -52.85           -0.9047           0.07092            0.04544             -35.93               -1.136
 10        0.02628        0.009369          -64.35            -1.216           0.06945            0.04022             -42.08               -1.561
 20        0.01027        0.007275          -29.17           -0.3002           0.05628            0.02931             -47.93               -2.095
 30        0.02992        0.006739          -77.48            -2.304           0.04372            0.02445             -44.08               -1.465
 40       0.006642        0.004351          -34.49           -0.2858           0.01393            0.01666              19

Saved: C:\Users\lpand003\Desktop\Research\ipa-pruning\physlab\SLP\SLP-CIFAR10\IPA_2nd_derivative_method\Comparison\ipa_compare_methods.png


Saved: C:\Users\lpand003\Desktop\Research\ipa-pruning\physlab\SLP\SLP-CIFAR10\IPA_2nd_derivative_method\Comparison\ipa_compare_differences.png
